# RQ I: Feedback mechanisms


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'rq1')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## Matched feedback comparisons


In [ ]:
from reported_runs.analysis.sections_rq1_3 import diagnostic_tables, horizon_counts, model_trajectories, advertising_display
feedback = trajectory.loc[trajectory.study.eq('movielens_feedback_a300_e804') & (~trajectory.stage.str.startswith('L3') | trajectory.user_update_policy.eq('full_reflection'))].copy()
feedback_effects = effect_summaries.loc[effect_summaries.study.eq('movielens_feedback_a300_e804') & effect_summaries.kind.isin(['feedback', 'individual'])].copy()
report.table(results['comparison_plan'].loc[results['comparison_plan'].study.str.startswith('movielens_') & results['comparison_plan'].kind.isin(['feedback', 'individual', 'sampling', 'advertising'])], 'rq1_matched_comparisons', "")

## Primary exposure effects


In [ ]:
figure = effect_grid(feedback_effects, ['head_exposure', 'weighted_head_exposure'], title='Added feedback mechanisms: matched exposure effects', scales=scales, value='value', facet='window')
report.figure(figure, feedback_effects, 'rq1_feedback_exposure_effects', "", recommended=False, why="")
report.table(effect_table(feedback_effects, 'head_exposure'), 'rq1_feedback_head_exposure', "")
report.table(effect_table(feedback_effects, 'weighted_head_exposure'), 'rq1_feedback_weighted_head_exposure', "")

In [ ]:
model_trajectories(report, feedback, ['head_exposure', 'weighted_head_exposure'], prefix='rq1_feedback_trajectory', context='Feedback-stage exposure trajectories')

## Selection relative to exposure, click behaviour, and coverage


In [ ]:
diagnostic_tables(report, feedback, {
    'click_exposure': ['head_click_share', 'click_exposure_gap', 'weighted_click_exposure_gap'],
    'click_behaviour': ['click_rate', 'no_click_rate', 'invalid_rate', 'valid_clicks'],
    'coverage': ['exposure_coverage', 'click_coverage'],
    'concentration': ['exposure_gini', 'click_gini']}, prefix='rq1_feedback', context="")

## Sampling feedback: matched 150/150 comparisons


In [ ]:
sampling = trajectory.loc[trajectory.study.eq('movielens_sampling_outcome_a150_r150_e804') & ~trajectory.condition_id.str.startswith('O')].copy()
sampling_effects = effect_summaries.loc[effect_summaries.kind.eq('sampling')].copy()
figure = effect_grid(sampling_effects, ['head_exposure', 'weighted_head_exposure'], title='Sampling feedback versus matched L3', scales=scales, value='value', facet='window')
report.figure(figure, sampling_effects, 'rq1_sampling_exposure_effects', "")
model_trajectories(report, sampling, ['head_exposure', 'weighted_head_exposure'], prefix='rq1_sampling_trajectory', context='Matched 150/150 sampling trajectories')

In [ ]:
population = label_conditions(results['population'].loc[results['population'].study.eq('movielens_sampling_outcome_a150_r150_e804') & ~results['population'].condition_id.str.startswith('O')])
figure = trajectory_grid(population, ['active_users', 'departures', 'replacements'], title='Sampling feedback: participation and turnover', scales=scales, series='condition_label', scale_overrides={'active_users': (0, 150), 'departures': (0, 20), 'replacements': (0, 20)})
report.figure(figure, population, 'rq1_sampling_population', "")
diagnostic_tables(report, population, {'population': ['active_users', 'departures', 'replacements']}, prefix='rq1_sampling', context="")
report.table(horizon_counts(population, ['departures', 'replacements']), 'rq1_sampling_turnover_totals', "")

In [ ]:
clusters = label_conditions(results['clusters'].loc[results['clusters'].study.eq('movielens_sampling_outcome_a150_r150_e804') & ~results['clusters'].condition_id.str.startswith('O')])
for condition, selected in clusters.groupby('condition_label', sort=False):
    figure = trajectory_grid(selected, 'active_share', title=f'Preference-cluster composition: {condition}', scales=scales, series='cluster')
    report.figure(figure, selected, 'rq1_clusters_' + selected.condition_id.iloc[0].lower().replace('__', '_'), "")
diagnostic_tables(report, clusters, {'cluster_composition': ['active_users', 'active_share']}, prefix='rq1_sampling', context="", extra_ids=['cluster'])

In [ ]:
from reported_runs.analysis.plotting import composition_grid
figure = composition_grid(clusters, title='Active preference-cluster composition: 100% stacked bars')
report.figure(figure, clusters, 'rq1_sampling_cluster_composition', "")

In [ ]:
diagnostic_tables(report, sampling, {
    'click_exposure': ['head_click_share', 'click_exposure_gap', 'weighted_click_exposure_gap'],
    'click_behaviour': ['click_rate', 'no_click_rate', 'valid_clicks'],
    'coverage': ['exposure_coverage', 'click_coverage'],
    'concentration': ['exposure_gini', 'click_gini']}, prefix='rq1_sampling', context="")

## Advertising and outcome feedback


In [ ]:
from reported_runs.analysis.advertising_slots import load_advertising_slot_metrics
from reported_runs.analysis.advertising_views import SLOT_METRICS, ACCOUNTING_LABELS, build_slot_views, slot_scales, slot_accounting_table
from reported_runs.analysis.sections_rq1_3 import diagnostic_table
from reported_runs.analysis.plotting import label
slot_sources = sources.loc[sources.study.eq('movielens_sampling_outcome_a150_r150_e804') & sources.condition_id.str.startswith('O')]
slot_metrics = load_advertising_slot_metrics(slot_sources, repo, repo / 'reported_runs/analysis/cache/advertising_slots')
slot_views = build_slot_views(slot_metrics, results['comparison_plan'])
slot_trajectory = label_conditions(slot_views['trajectory'])
slot_effects = slot_views['effect_summaries']
advertising_scales = slot_scales(scales, slot_views)
figure = effect_grid(slot_effects, SLOT_METRICS, title='Advertising exposure effects: displayed ranks 2–10 only', scales=advertising_scales, value='value', facet='window')
figure._analysis_spec['exposure_scope'] = 'Displayed ranks 2–10 only, in every condition; original rank discounts retained'
report.figure(figure, slot_effects, 'rq1_advertising_ranks_2_10_effects', "", recommended=False, why="")
for metric in SLOT_METRICS:
    table = effect_table(slot_effects, metric).rename(columns={'value': label(metric) + ' difference (pp)'})
    report.table(table, 'rq1_advertising_' + metric + '_effects', "", source_frame=slot_metrics)
report.table(diagnostic_table(slot_trajectory, SLOT_METRICS), 'rq1_advertising_ranks_2_10_levels', "", source_frame=slot_metrics)

In [ ]:
figure = trajectory_grid(slot_trajectory, SLOT_METRICS, title='Advertising exposure trajectories: displayed ranks 2–10 only', scales=advertising_scales, series='condition_label')
figure._analysis_spec['exposure_scope'] = 'Displayed ranks 2–10 only, in every condition; original rank discounts retained'
report.figure(figure, slot_trajectory, 'rq1_advertising_ranks_2_10_trajectories', "")
slot_consistency = slot_views['seed_consistency'].copy()
slot_consistency['metric'] = slot_consistency.metric.map(label)
report.table(slot_consistency, 'rq1_advertising_ranks_2_10_seed_consistency', "", columns=['comparison', 'metric', 'window', 'n_seed_blocks', 'mean_effect', 'min_effect', 'max_effect', 'same_nonzero_sign'], source_frame=slot_metrics)
slot_accounting = label_conditions(slot_accounting_table(slot_views['summaries']))
report.table(slot_accounting, 'rq1_advertising_rank1_inclusion_accounting', "", columns=['condition_label', 'seed', 'population_seed', 'window', *ACCOUNTING_LABELS.values()], source_frame=slot_metrics)

In [ ]:
advertising_effects = effect_summaries.loc[effect_summaries.kind.eq('advertising')].copy()
figure = effect_grid(advertising_effects, ['head_exposure', 'weighted_head_exposure'], title='Advertising exposure effects: all displayed ranks 1–10', scales=scales, value='value', facet='window')
figure._analysis_spec['exposure_scope'] = 'All displayed ranks 1–10, including any sponsored rank-1 item'
report.figure(figure, advertising_effects, 'rq1_advertising_exposure_effects', "")
for metric in ['head_exposure', 'weighted_head_exposure']:
    table = effect_table(advertising_effects, metric).rename(columns={'value': label(metric) + ' difference (displayed ranks 1–10; pp)'})
    report.table(table, 'rq1_advertising_' + metric + '_ranks_1_10_effects', "")
advertising_raw = label_conditions(results['advertising'].loc[results['advertising'].study.eq('movielens_sampling_outcome_a150_r150_e804') & results['advertising'].condition_id.str.startswith('O')])
advertising = advertising_display(advertising_raw)

In [ ]:
figure = trajectory_grid(advertising, ['advertising_eligible_users', 'sponsored_exposures', 'sponsored_clicks'], title='Advertising delivery and sponsored responses', scales=scales, series='condition_label', scale_overrides={'advertising_eligible_users': (0, 150), 'sponsored_exposures': (0, 150), 'sponsored_clicks': (0, 70)})
report.figure(figure, advertising, 'rq1_advertising_delivery', "")
diagnostic_tables(report, advertising, {'eligibility': ['advertising_eligible_users', 'advertising_eligibility_share'], 'sponsored_counts': ['sponsored_exposures', 'sponsored_clicks'], 'sponsored_shares': ['sponsored_exposure_share', 'weighted_sponsored_exposure', 'sponsored_click_share']}, prefix='rq1_advertising', context="")
report.table(horizon_counts(advertising, ['sponsored_exposures', 'sponsored_clicks']), 'rq1_advertising_totals', "")

In [ ]:
advertising_outcomes = trajectory.loc[trajectory.study.eq('movielens_sampling_outcome_a150_r150_e804') & trajectory.condition_id.str.startswith('O')]
diagnostic_tables(report, advertising_outcomes, {
    'click_exposure': ['head_click_share', 'click_exposure_gap', 'weighted_click_exposure_gap'],
    'click_behaviour': ['click_rate', 'no_click_rate', 'valid_clicks'],
    'coverage': ['exposure_coverage', 'click_coverage'],
    'concentration': ['exposure_gini', 'click_gini']}, prefix='rq1_advertising', context="")

## Descriptive mean and consistency across available seeds


In [ ]:
from reported_runs.analysis.presentation import cross_seed_table
for metric in ['head_exposure', 'weighted_head_exposure']:
    table = cross_seed_table(results, 'rq1', [metric])
    report.table(table, f'rq1_seed_consistency_{metric}', "")